# DataSage 0/4 — W&B Training Data Export

Exports training metrics from 3 GRPO training runs. **No GPU required.**

| Setting | Value |
|---------|-------|
| Runtime | CPU (no GPU needed) |
| Secrets | `WANDB_API_KEY` |
| Output | `wandb_training_data.json` |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ricalanis/openenv-hackaton/blob/main/demo/inference/00_wandb_export.ipynb)

In [ ]:
# Cell 1: Install dependencies
!pip install -q wandb numpy

In [ ]:
# Cell 2: API Keys + Colab detection
import os
import sys

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["WANDB_API_KEY"] = userdata.get("WANDB_API_KEY")
        print("Loaded WANDB_API_KEY from Colab Secrets")
    except Exception:
        print("WARNING: WANDB_API_KEY not found in Colab Secrets.")
        print("Go to: left sidebar > Secrets (key icon) > Add WANDB_API_KEY")
else:
    print("Running locally — using environment variables")

assert os.environ.get("WANDB_API_KEY"), (
    "WANDB_API_KEY not set. "
    "In Colab: add it to Secrets (key icon in left sidebar). "
    "Locally: export WANDB_API_KEY=..."
)
print(f"WANDB_API_KEY: ...{os.environ['WANDB_API_KEY'][-6:]}")

In [ ]:
# Cell 3: Configuration
#
# W&B project: https://wandb.ai/ricalanis/datasage
# These are the 3 GRPO training runs for each DataSage stage.

WANDB_PROJECT = "ricalanis/datasage"

WANDB_RUNS = {
    "cleaning":   "xuwyjpe6",   # GRPO cleaning training run
    "enrichment": "orww3s2q",   # GRPO enrichment training run
    "answering":  "2mltqk5w",   # GRPO answering training run
}

print("Will fetch runs:")
for task, run_id in WANDB_RUNS.items():
    print(f"  {task}: {WANDB_PROJECT}/{run_id}")

In [ ]:
# Cell 4: Fetch W&B data
import json
import numpy as np
import wandb

api = wandb.Api()
wandb_data = {}

for task, run_id in WANDB_RUNS.items():
    print(f"\n{'='*60}")
    print(f"Fetching: {task} ({run_id})")
    print(f"{'='*60}")

    try:
        run = api.run(f"{WANDB_PROJECT}/{run_id}")
        print(f"  Name:  {run.name}")
        print(f"  State: {run.state}")

        # Fetch full history
        history = [dict(row) for row in run.scan_history()]
        print(f"  History rows: {len(history)}")

        # Build metric series — only numeric values
        metrics = {}
        all_keys = set()
        for row in history:
            all_keys.update(row.keys())

        for key in sorted(all_keys):
            values = [row.get(key) for row in history if row.get(key) is not None]
            if values and all(isinstance(v, (int, float)) for v in values):
                metrics[key] = {
                    "values": [float(v) for v in values],
                    "min": float(min(values)),
                    "max": float(max(values)),
                    "mean": float(np.mean(values)),
                    "final": float(values[-1]),
                    "n_points": len(values),
                }

        wandb_data[task] = {
            "run_id": run_id,
            "run_name": run.name,
            "state": run.state,
            "config": dict(run.config) if run.config else {},
            "summary": {
                k: v for k, v in run.summary.items()
                if isinstance(v, (int, float, str, bool))
            },
            "metrics": metrics,
            "n_history_rows": len(history),
        }

        # Print key metrics
        for key in sorted(metrics.keys()):
            if any(kw in key.lower() for kw in ["reward", "loss"]):
                m = metrics[key]
                print(f"  {key}: mean={m['mean']:.4f}, final={m['final']:.4f}")

    except Exception as e:
        print(f"  ERROR: {e}")
        wandb_data[task] = {"run_id": run_id, "error": str(e)}

print(f"\nDone. Tasks fetched: {list(wandb_data.keys())}")

In [ ]:
# Cell 5: Save to JSON and download
import os
import sys
import json
import numpy as np

IN_COLAB = "google.colab" in sys.modules or os.path.exists("/content")

def _json_safe(obj):
    """Handle numpy types for JSON serialization."""
    if isinstance(obj, (np.integer,)): return int(obj)
    if isinstance(obj, (np.floating,)): return float(obj)
    if isinstance(obj, np.ndarray): return obj.tolist()
    return obj

OUTPUT_FILE = "wandb_training_data.json"

with open(OUTPUT_FILE, "w") as f:
    json.dump(wandb_data, f, indent=2, default=_json_safe)

size_kb = os.path.getsize(OUTPUT_FILE) / 1024
print(f"Saved: {OUTPUT_FILE} ({size_kb:.1f} KB)")

if IN_COLAB:
    from google.colab import files
    files.download(OUTPUT_FILE)
    print("Downloaded to your browser.")
else:
    print(f"File at: {os.path.abspath(OUTPUT_FILE)}")